# 영화 정보 구조화 — OpenAI 클라이언트 버전

영화 제목 입력 → **OpenAI 내장 web_search 툴로 검색** → **Structured Output**으로 공통 `Movie` 스키마에 맞춰 구조화한다.

`responses.parse(tools=[web_search], text_format=Movie)` 한 번의 호출로 **검색과 구조화가 동시에** 일어난다.

> 사전 준비: `.env` 에 `OPENAI_API_KEY` 만 있으면 된다 (검색 키 불필요). 실행: `uv run jupyter lab`.

## 1. 환경 설정

In [5]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트(schemas.py 가 있는 곳)를 import 경로에 추가
ROOT = Path.cwd()
if not (ROOT / "schemas.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

TITLE = "헤일메리"  # ← 원하는 영화 제목으로 변경
MODEL = "gpt-5.4-nano"  # web_search + structured output 지원
EFFORT = "low"          # reasoning effort (low: 빠르고 저렴)
print("ROOT:", ROOT, "| TITLE:", TITLE, "| MODEL:", MODEL, "| effort:", EFFORT)

ROOT: C:\workspace\rnd-onboarding | TITLE: 헤일메리 | MODEL: gpt-5.4-nano | effort: low


## 2. 검색 + 구조화 (단일 호출)

`tools=[{"type": "web_search"}]` 로 모델이 직접 웹을 검색하고, `text_format=Movie` 로 결과를 `Movie` 스키마에 맞춰 돌려준다.

In [6]:
from openai import OpenAI
import prompts
from schemas import Movie

client = OpenAI()

msgs = prompts.render("v2", TITLE)  # findings 없음 → 모델이 web_search 로 직접 검색
resp = client.responses.parse(
    model=MODEL,
    tools=[{"type": "web_search"}],
    reasoning={"effort": EFFORT},
    input=[
        {"role": "system", "content": msgs["system"]},
        {"role": "user", "content": msgs["user"]},
    ],
    text_format=Movie,
)
movie = resp.output_parsed
print(movie.pretty())

🎬 프로젝트 헤일메리  (2026-03-18)
⭐ 평점      : 8.2
🎥 감독      : 필 로드, 크리스 밀러
🏢 제작/배급 : MGM
👥 출연      : 라이언 고슬링(라일랜드 그레이스), 산드라 휠러(Eva Stratt), James Ortiz(Rocky), 라이오넬 보이스(Carl), 켄 렁(Yao)
📖 줄거리    : 라일랜드 그레이스는 뜻밖의 친구와 함께, 죽어가는 태양을 구하기 위한 마지막 임무를 수행하게 된다. 기억을 잃은 채 우주 한복판에서 깨어난 그레이스는 인류를 구할 미션을 향해 필사의 여정을 이어간다.


## 3. 프롬프트 매니징 — v1 vs v2 비교

같은 영화에 프롬프트 버전만 바꿔 끼우며 출력 차이를 비교한다 (`prompts` 모듈이 버전 관리).

In [3]:
for v in prompts.list_versions():
    msgs = prompts.render(v, TITLE)
    resp = client.responses.parse(
        model=MODEL,
        tools=[{"type": "web_search"}],
        reasoning={"effort": EFFORT},
        input=[
            {"role": "system", "content": msgs["system"]},
            {"role": "user", "content": msgs["user"]},
        ],
        text_format=Movie,
    )
    print(f"=== prompt {v} ===")
    print(resp.output_parsed.model_dump_json(indent=2))
    print()

=== prompt v1 ===
{
  "title": "인터스텔라",
  "release_date": "2014-11-07",
  "production": "크리스토퍼 놀런 감독(제작/각본), 레전더리 픽처스(Legendary Pictures), 워너 브라더스(워너 브라더스 픽처스/Warner Bros.) 제작",
  "director": "크리스토퍼 놀런",
  "cast": [
    "매튜 맥커너히",
    "앤 해서웨이",
    "제시카 차스테인",
    "마이클 케인",
    "카일리 머레이"
  ],
  "synopsis": "지구가 점점 황폐해지자, 인류의 생존을 위해 한 팀이 웜홀 너머의 새로운 거주 가능 행성을 찾아 장대한 우주 탐사에 나선다. 시간과 중력의 극한 속에서 가족과 선택의 무게가 뒤엉키며, 임무의 목적이 점차 더 깊은 의미로 드러난다.",
  "rating": 8.6
}



=== prompt v2 ===
{
  "title": "인터스텔라",
  "release_date": "2014-11-05",
  "production": "Legendary Pictures",
  "director": "크리스토퍼 놀란",
  "cast": [
    "매튜 맥커너히",
    "앤 해서웨이",
    "제시카 차스테인",
    "마이클 케인",
    "티모시 샬라메"
  ],
  "synopsis": "지구가 극심한 재난과 식량 부족으로 붕괴해 가는 미래, 인류를 위한 새로운 거처를 찾기 위해 우주 비행 임무가 시작된다. 전(前) NASA 조종사 쿠퍼는 가족을 지키려는 마음과 인류의 생존을 위한 결단 사이에서 선택을 강요받는다. 임무는 우주를 향한 탐색을 넘어, 시간과 중력의 미스터리를 마주하며 점점 더 큰 대가를 요구한다.",
  "rating": 8.7
}



## 4. 정리

- OpenAI Responses API 는 **툴 사용(web_search) + 구조화 출력(text_format)** 을 한 호출에 융합한다.
- 반환 `resp.output_parsed` 가 곧 `Movie` 인스턴스 → 후처리 불필요.
- LangChain 버전(`02_langchain_version.ipynb`)은 같은 일을 2단계 체인으로 한다 — 비교해 볼 것.